L4: Tools for a Customer Outreach Campaign
In this lesson, you will learn more about Tools. You'll focus on three key elements of Tools:

- Versatility
- Fault Tolerance
- Caching

In [8]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [9]:
from crewai import Agent, Task, Crew

In [10]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))


True

In [11]:
import sys
import os
from utils.api_utils import get_openai_api_key, pretty_print_result
from utils.api_utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [12]:
from crewai import Agent,Task,Crew
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match "
         "our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    verbose=True
)

In [13]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of CrewAI's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    verbose=True
)

In [14]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [15]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

In [16]:
from crewai.tools import BaseTool


- Every Tool needs to have a name and a description.
- For simplicity and classroom purposes, SentimentAnalysisTool will return positive for every text.
- When running locally, you can customize the code with your logic in the _run function.


In [17]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [18]:
sentiment_analysis_tool = SentimentAnalysisTool()

In [19]:
lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

The Personalized Outreach Task is using your custom Tool SentimentAnalysisTool, as well as crewAI's SerperDevTool (search_tool).

In [20]:
personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

In [21]:
#creating the crew
from crewai import Crew
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=True,
	memory=True
)

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()

# Optional: confirm key is loaded
print("OpenAI Key Found:", os.getenv("OPENAI_API_KEY") is not None)


OpenAI Key Found: True


In [23]:

inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Online Learning Platform",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 22a68b30-9149-487f-a4ea-dbb961f0db22                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: bddaed42-689f-47ce-8209-10213ab79a26
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: bddaed42-689f-47ce-8209-10213ab79a26
       Status: Executing Task...
    └── 🤖 Agent: Sales Representative
            Status: In Progress

# Agent: Sales Representative
## Task: Conduct an in-depth analysis of DeepLearningAI, a company in the Online Learning Platform sector that recently showed interest in our solutions. Utilize all available data sources to compile a detailed profile, focusing on key decision-makers, recent business developments, and potential needs that align with our offerings. This task is crucial for tailoring our engagement strategy effectively.
Don't make assumptions and only use information you absolutely sure about.


🚀 Crew: crew
└── 📋 Task: bddaed42-689f-47ce-8209-10213ab79a26
       Status: Executing Task...
    └── 🤖 Agent: Sales Representative
            Status: In Progress
        └── 🧠 Thinking...

...



# Agent: Lead Sales Representative
## Final Answer: 
Subject: Exploring Synergies Between DeepLearningAI and [Your Company Name]

Dear Andrew Ng,

I hope this message finds you well. As a professional deeply immersed in the realm of artificial intelligence, I understand the significance of harnessing cutting-edge technologies and fostering innovation to propel organizations forward. It is with great admiration that I reach out to you to explore potential synergies between DeepLearningAI and [Your Company Name].

While I regrettably couldn't access specific files pertaining to DeepLearningAI's recent advancements, the remarkable strides your esteemed organization has made in AI have not gone unnoticed. I was particularly impressed by the recent article on The Batch highlighting a drone from Skyfire AI saving a police officer's life—a testament to the practical applications and impact AI can have in real-world scenarios.

Our solutions at [Your Company Name] are specifically designed t

🚀 Crew: crew
├── 📋 Task: bddaed42-689f-47ce-8209-10213ab79a26
│      Assigned to: Sales Representative
│      Status: ✅ Completed
│   └── 🤖 Agent: Sales Representative
│           Status: ✅ Completed
└── 📋 Task: 36c2c97b-3e1a-4d49-8414-1f99a68bf2f0
       Status: Executing Task...
    └── 🤖 Agent: Lead Sales Representative
            Status: ✅ Completed
        ├── 🔧 Used Sentiment Analysis Tool (1)
        └── 🔧 Used Search the internet with Serper (1)

🚀 Crew: crew
├── 📋 Task: bddaed42-689f-47ce-8209-10213ab79a26
│      Assigned to: Sales Representative
│      Status: ✅ Completed
│   └── 🤖 Agent: Sales Representative
│           Status: ✅ Completed
└── 📋 Task: 36c2c97b-3e1a-4d49-8414-1f99a68bf2f0
       Assigned to: Lead Sales Representative
       Status: ✅ Completed
    └── 🤖 Agent: Lead Sales Representative
            Status: ✅ Completed
        ├── 🔧 Used Sentiment Analysis Tool (1)
        └── 🔧 Used Search the internet with Serper (1)

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 36c2c97b-3e1a-4d49-8414-1f99a68bf2f0                                                                     │
│  Agent: Lead Sales Representative                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 22a68b30-9149-487f-a4ea-dbb961f0db22                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
from IPython.display import Markdown

Markdown(str(result))


Subject: Exploring Synergies Between DeepLearningAI and [Your Company Name]

Dear Andrew Ng,

I hope this message finds you well. As a professional deeply immersed in the realm of artificial intelligence, I understand the significance of harnessing cutting-edge technologies and fostering innovation to propel organizations forward. It is with great admiration that I reach out to you to explore potential synergies between DeepLearningAI and [Your Company Name].

While I regrettably couldn't access specific files pertaining to DeepLearningAI's recent advancements, the remarkable strides your esteemed organization has made in AI have not gone unnoticed. I was particularly impressed by the recent article on The Batch highlighting a drone from Skyfire AI saving a police officer's life—a testament to the practical applications and impact AI can have in real-world scenarios.

Our solutions at [Your Company Name] are specifically designed to complement and amplify the efforts of pioneering companies like DeepLearningAI. By leveraging our expertise and resources, we aim to contribute to the advancement of AI applications and the realization of our shared goals.

I firmly believe that a collaborative partnership between our entities holds immense potential to drive technological progress and unlock new opportunities for growth. Together, we can propel innovation in the AI landscape and create a profound impact on the industry.

I would be delighted to engage in further discussions on how our offerings can directly support DeepLearningAI's pursuit of innovation and success. Your vision and insights are highly esteemed, and I look forward to exploring how we can collaboratively shape the future of AI.

Thank you for considering this mutual journey towards excellence. I am excited about the prospect of working together and contributing to the incredible narrative of DeepLearningAI.

Warm regards,

[Your Name]
[Your Position]
[Your Contact Information]